In [1]:
import os

In [2]:
%pwd

'd:\\Coding\\Data science\\Kidney-Disease-Classification\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Coding\\Data science\\Kidney-Disease-Classification'

In [5]:
import dagshub
import mlflow

# This automatically handles both the tracking URI and authentication
dagshub.init(repo_owner="Manthan007", repo_name="Kidney-Disease-Classification", mlflow=True)

c:\Users\admin\miniconda3\envs\kidney\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as Manthan007

Initialized MLflow to track repo "Manthan007/Kidney-Disease-Classification"

Repository Manthan007/Kidney-Disease-Classification initialized!

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_to_model: Path
    training_data: Path
    all_params: dict   
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import *

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        training = self.config.training
        trained_path = self.config.data_ingestion
        eval_config = EvaluationConfig(
            path_to_model=Path(training.trained_model_path),
            training_data=Path(trained_path.root_dir),
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_to_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    

    def log_into_mlflow(self):
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )

            # 3. Log/Register the model
            mlflow.keras.log_model(
                self.model, 
                "model", 
                registered_model_name="VGG16Model"
            )

In [12]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 150 images belonging to 2 classes.
10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.7267 - loss: 1.0784


2026/06/09 13:38:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 13:38:10 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/06/09 13:39:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


🏃 View run languid-toad-159 at: https://dagshub.com/Manthan007/Kidney-Disease-Classification.mlflow/#/experiments/0/runs/428e6db60b9244a4845bf40c962c2c5e
🧪 View experiment at: https://dagshub.com/Manthan007/Kidney-Disease-Classification.mlflow/#/experiments/0
